## MSPE fine-tuning Attempt 1 WEIGHT LAMBDA

This notebook produces the Attempt 1 fine-tuning experiments for:

A pretrained SwinUNETR (V2) backbone is frozen, and only the MSPE patch embedding is trained.


1. MSPE naive routing -- identical kernels with resolution conditional routing.
2. MSPE overlapping kernels -- overlapping kernels at stride v2 (i+2).
3. MSPE dilating kernels -- identical kernel sizes with different dilation rates.

Vessels Dataset: 
https://figshare.com/s/0325c9e375930ef76795?file=62063935

In [ ]:
!python -c "import monai" || pip install -q "monai[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
!python -c "import wandb" || pip install -q wandb
%matplotlib inline

In [ ]:
import sys
import os

colab = True
if colab:
    from google.colab import drive, runtime, userdata
    drive.mount('/content/drive')
    dataset_dir = r"/content/drive/MyDrive/DATASETS/HISTOPOLOGY/"
    root_dir = r"/content/drive/MyDrive/RUNS/HISTOPOLOGY/ABLATION/"
    mspe_code_dir = r"/content/drive/MyDrive/MSPE/"
    sys.path.append(mspe_code_dir)
    num_workers = 12
    cache_rate = 1.0
else:
    dataset_dir = r""
    root_dir = r""
    mspe_code_dir = r""
    sys.path.append(mspe_code_dir)
    num_workers = 0
    cache_rate = 0.0
    userdata = None

os.makedirs(root_dir, exist_ok=True)

In [ ]:
import glob
import os
import time
import random
import copy
from abc import ABC, abstractmethod

import nibabel as nib
import numpy as np
import monai
import wandb
import tqdm
import matplotlib.pyplot as plt

from monai.apps import CrossValidation
from monai.config import print_config
from monai.data import CacheDataset, Dataset, create_test_image_3d, decollate_batch, DataLoader, list_data_collate, PILReader, ThreadDataLoader
from monai.data.utils import partition_dataset
from monai.handlers import (
    MeanDice,
    MLFlowHandler,
    StatsHandler,
    TensorBoardImageHandler,
    TensorBoardStatsHandler,
)
from monai.inferers import sliding_window_inference
from monai.losses import TverskyLoss, DiceLoss, DiceCELoss, MaskedDiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.networks.blocks import PatchEmbed
from monai.networks.layers import trunc_normal_, Conv, Norm
from monai.networks.nets import UNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    RandAffined, RandFlipd,RandRotated, RandScaleIntensityd, RandAdjustContrastd,
    DivisiblePadd, CropForegroundd,
    ScaleIntensityRanged, ScaleIntensityRangePercentilesd,
    AsDiscrete, AsDiscreted, CastToTyped, ToTensord,
)
from monai.utils import first, set_determinism

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from swin_mspe import (
    MSPEPatchEmbedSwinNaiveRouting,
    MSPEPatchEmbedSwinDilatingK3,
    MSPEPatchEmbedSwinOverlapping, 
    DEFAULT_RESOLUTIONS,
    DEFAULT_K,
    mspe_swin_train_step,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bfloat16 on Ampere and later
AMP_DTYPE = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"AMP dtype: {AMP_DTYPE}")
print_config()

In [ ]:
full_dataset = []
images_list = glob.glob(dataset_dir + "/raw_images/*.png")
# NOTE: Expert 1, vascular wall annotations
masks_list = glob.glob(dataset_dir + "/expert_1/vascular_wall/*.png")
roi_masks_list = glob.glob(dataset_dir + "/roi_masks/*.png")

# Sort images, masks for reproducibility
images_list.sort()
masks_list.sort()
roi_masks_list.sort()

assert len(images_list) == len(masks_list) == len(roi_masks_list), \
    f"Mismatch: {len(images_list)} images, {len(masks_list)} masks, {len(roi_masks_list)} ROI masks"

for img, mask, roi in tqdm.tqdm(zip(images_list, masks_list, roi_masks_list), total=len(images_list)):
    full_dataset.append({
        "image": img,
        "label": mask,
        "roi_mask": roi,
    })

print(f"Total samples: {len(full_dataset)}")


# Set fold for 5-fold cross-validation 

class CVDataset(Dataset):

    def __init__(self, data, cache_rate=cache_rate, num_workers=num_workers, transform=None):
        data = self._split_datalist(datalist=data)
        super().__init__(data=data, transform=transform)

    def _split_datalist(self, datalist):
        raise NotImplementedError(f"Subclass {self.__class__.__name__} must implement this method.")


FOLD_NUMBER = 0
total_folds = 5
folds = list(range(total_folds))

cv_splitter = CrossValidation(
    dataset_cls=CVDataset,
    data=full_dataset,
    nfolds=total_folds,
    seed=2026,
)

train_fold_ds = cv_splitter.get_dataset(
    folds=folds[:FOLD_NUMBER] + folds[FOLD_NUMBER + 1:],
    transform=None,
)
val_fold_ds = cv_splitter.get_dataset(
    folds=FOLD_NUMBER,
    transform=None,
)

train_dataset = list(train_fold_ds.data)
val_dataset = list(val_fold_ds.data)

print(f"Fold {FOLD_NUMBER}: Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadImaged(keys=["label", "roi_mask"], reader=PILReader(converter=lambda image: image.convert("L"))),
    AsDiscreted(keys=["label", "roi_mask"], threshold=127),
    EnsureChannelFirstd(keys=["image", "label", "roi_mask"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=["image", "label", "roi_mask"]),
    CastToTyped(keys=["image", "label", "roi_mask"], dtype=(torch.float32, torch.uint8, torch.uint8)),
    
    # MSPE augmentation pipeline, with low probs 
    RandFlipd(keys=["image", "label", "roi_mask"], prob=0.2, spatial_axis=1), 
    RandFlipd(keys=["image", "label", "roi_mask"], prob=0.2, spatial_axis=0), 
    RandRotated(
        keys=["image", "label", "roi_mask"],
        prob=0.2,
        range_x=np.deg2rad(15),
        mode=("bilinear", "nearest", "nearest"),
        padding_mode="zeros",
    ),
    RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.1),      
    RandAdjustContrastd(keys=["image"], gamma=(0.9, 1.1), prob=0.1),
    # For Swin
    DivisiblePadd(keys=["image", "label", "roi_mask"], k=32, mode="constant"),
])

val_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadImaged(keys=["label", "roi_mask"], reader=PILReader(converter=lambda image: image.convert("L"))),
    AsDiscreted(keys=["label", "roi_mask"], threshold=127),
    EnsureChannelFirstd(keys=["image", "label", "roi_mask"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=["image", "label", "roi_mask"]),
    CastToTyped(keys=["image", "label", "roi_mask"], dtype=(torch.float32, torch.uint8, torch.uint8)),

    # For Swin
    DivisiblePadd(keys=["image", "label", "roi_mask"], k=32, mode="constant"),  
])

train_ds = CacheDataset(data=train_dataset, transform=train_transforms,
                        cache_rate=cache_rate, num_workers=num_workers)
train_loader = ThreadDataLoader(train_ds, num_workers=num_workers,
                                batch_size=1, shuffle=True,
                                pin_memory=True, persistent_workers=False,
                                collate_fn=list_data_collate)

val_ds = CacheDataset(data=val_dataset, transform=val_transforms,
                      cache_rate=cache_rate, num_workers=num_workers)
val_loader = ThreadDataLoader(val_ds, num_workers=num_workers,
                              batch_size=1, shuffle=False)

# Test is the same as val for cross-val
test_ds = val_ds
test_loader = val_loader

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
def display_sample_pairs(data_loader, n_samples=3):
    fig, axes = plt.subplots(n_samples, 4, figsize=(20, 4 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for idx, batch in enumerate(data_loader):
        if idx >= n_samples:
            break

        image = batch["image"][0].cpu().permute(1, 2, 0)
        label = batch["label"][0][0].cpu()
        roi_mask = batch["roi_mask"][0][0].cpu()
        filename = os.path.basename(batch["image"].meta["filename_or_obj"][0])

        ax_row = axes[idx]
        ax_row[0].imshow(image)
        ax_row[0].set_title(f"Image\n{filename}", fontsize=9)
        ax_row[0].axis("off")
        ax_row[1].imshow(label, cmap="gray")
        ax_row[1].set_title(f"Mask (vascular wall)\n{filename}", fontsize=9)
        ax_row[1].axis("off")
        ax_row[2].imshow(roi_mask, cmap="gray")
        ax_row[2].set_title(f"ROI Mask\n{filename}", fontsize=9)
        ax_row[2].axis("off")
        ax_row[3].imshow(image, cmap="gray")
        ax_row[3].imshow(roi_mask, cmap="Blues", alpha=0.35)
        ax_row[3].imshow(label, cmap="Reds", alpha=0.45)
        ax_row[3].set_title(f"Mask + ROI Overlay\n{filename}", fontsize=9)
        ax_row[3].axis("off")

    plt.tight_layout()
    plt.show()

# Take a look at train data 
display_sample_pairs(train_loader, n_samples=3)

In [ ]:
# Experimental config

FEATURE_SIZE = 24
MAX_EPOCHS = 10
VAL_INTERVAL = 2
EARLY_STOPPING_PATIENCE = 1000 # stop train if no improvement
BATCH_SIZE = 1

# Warm-start each MSPE kernel from the pretrained baseline patch_embed 
WARM_START = True

# Validate (and select the best checkpoint) across these effective resolutions (OFF)
VAL_SELECT_RESOLUTIONS = [None] 

# Optimizer 
LR = 1e-3
WD = 1e-5
WANDB_PROJECT = "VESSELS_MSPE_SWIN_ALL_FINE_TUNING"
# train/test res are [256, 384, 512, 768, 1024, 1280]
TEST_RESOLUTIONS = list(DEFAULT_RESOLUTIONS)
FOLD_TAG = f"FOLD_{FOLD_NUMBER}"

# Pretrained baseline checkpoint (SwinUNETR V2, the same fold split)
BASELINE_CHECKPOINT = r"/content/drive/MyDrive/RUNS/HISTOPOLOGY/FULL_TRAINING_SWIN/best_metric_BASELINE_SWIN_V2_FOLD_0.pth"

post_pred = Compose([AsDiscrete(argmax=True, to_onehot=2)])
post_label = Compose([AsDiscrete(to_onehot=2)])

#  MSPE variants to fine-tune
# lambda ablation (Table tab:lambda): routing only, sweep native-resolution loss weight lambda
LAMBDAS = [0.0, 0.5, 1.0]
EXPERIMENTS = [
    {
        "name": f"FT_ROUTING_LAM{int(lam*10):02d}",  # LAM00 / LAM05 / LAM10
        "mspe_class": MSPEPatchEmbedSwinNaiveRouting,
        "description": f"Frozen backbone + naive routing (lambda={lam})",
        "lam": lam,
    }
    for lam in LAMBDAS
]

print(f"Experiments to run: {[e['name'] for e in EXPERIMENTS]}")
print(f"Baseline checkpoint: {BASELINE_CHECKPOINT}")
print(f"Fold: {FOLD_TAG}, epochs: {MAX_EPOCHS}, LR: {LR}")
print(f"Warm-start MSPE kernels from baseline: {WARM_START}")
print(f"Checkpoint selection resolutions: {VAL_SELECT_RESOLUTIONS}")

In [ ]:
# Helpers

from swin_mspe import pi_resize

# WB login
def login_wandb():
    if colab:
        wandb.login(key=userdata.get("WANDB_API_KEY"))


# Looad only the frozen backbone from the baseline checkpoint
def _load_baseline_backbone_only(model, baseline_checkpoint):
    checkpoint = torch.load(baseline_checkpoint, weights_only=True)

    # Skip baseline embedding keys so MSPE starts fresh
    backbone_state = {
        name: tensor
        for name, tensor in checkpoint.items()
        if not name.startswith("swinViT.patch_embed.")
    }
    skipped_patch_keys = len(checkpoint) - len(backbone_state)

    incompatible = model.load_state_dict(backbone_state, strict=False)
    unexpected_keys = list(incompatible.unexpected_keys)
    missing_non_patch_keys = [
        name for name in incompatible.missing_keys
        if not name.startswith("swinViT.patch_embed.")
    ]

    # Fail if anything except patch_embed was skipped
    if unexpected_keys or missing_non_patch_keys:
        raise RuntimeError(
            "Backbone-only checkpoint load failed. "
            f"Unexpected keys: {unexpected_keys}; missing non-patch keys: {missing_non_patch_keys}"
        )

    print(f"Loaded baseline backbone from {baseline_checkpoint}")
    print(f"Skipped baseline patch_embed keys: {skipped_patch_keys}")


# Warm-start MSPE kernels from the pretrained baseline patch_embed (instead of random init)
def _warm_start_mspe_from_baseline(mspe_embed, baseline_checkpoint, device):
    """Initialise every MSPE kernel from the pretrained baseline 2x2 patch_embed. """
    if not hasattr(mspe_embed, "patch_kernels"):
        print("Embedding has no patch_kernels; skipping warm-start.")
        return

    checkpoint = torch.load(baseline_checkpoint, weights_only=True)
    proj_w = checkpoint["swinViT.patch_embed.proj.weight"].to(device)  # [embed_dim, in_chans, 2, 2]
    proj_b = checkpoint["swinViT.patch_embed.proj.bias"].to(device)
    base_size = tuple(proj_w.shape[2:])

    with torch.no_grad():
        for k in range(len(mspe_embed.patch_kernels)):
            kernel = mspe_embed.patch_kernels[k]
            k_size = tuple(kernel.weight.shape[2:])
            if k_size == base_size:
                kernel.weight.copy_(proj_w)                                      # exact (routing 2x2)
            else:
                kernel.weight.copy_(pi_resize(proj_w, list(k_size)).to(device))  # PI-resize to k_size
            if kernel.bias is not None:
                kernel.bias.copy_(proj_b)
        # warm-start the embedding norm affine as well
        if mspe_embed.norm is not None and "swinViT.patch_embed.norm.weight" in checkpoint:
            mspe_embed.norm.weight.copy_(checkpoint["swinViT.patch_embed.norm.weight"].to(device))
            mspe_embed.norm.bias.copy_(checkpoint["swinViT.patch_embed.norm.bias"].to(device))

    print(f"Warm-started {len(mspe_embed.patch_kernels)} MSPE kernels from baseline patch_embed "
          f"(exact copy where kernel size == {base_size}, PI-resize otherwise).")


# Get the baseline model for eval
def create_baseline_model(baseline_checkpoint, device):
    model = monai.networks.nets.SwinUNETR(
        in_channels=3,
        out_channels=2,
        spatial_dims=2,
        feature_size=FEATURE_SIZE,
        use_v2=True,
    ).to(device)

    # Baseline embedding --as in default from monai
    model.swinViT.patch_embed = PatchEmbed(
        patch_size=2,
        in_chans=3,
        embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm,
        spatial_dims=2,
    ).to(device)

    model.load_state_dict(torch.load(baseline_checkpoint, weights_only=True))
    model.eval()
    print(f"Loaded baseline checkpoint for evaluation from {baseline_checkpoint}")
    return model

# Get all exp models
def create_model_for_variant(variant_class, baseline_checkpoint, device):

    # Create standard SwinUNETR
    model = monai.networks.nets.SwinUNETR(
        in_channels=3,
        out_channels=2,
        spatial_dims=2,
        feature_size=FEATURE_SIZE,
        use_v2=True,
    ).to(device)

    # Load only baseline backbone weights -- skip baseline patch_embed
    _load_baseline_backbone_only(model, baseline_checkpoint)

    #  Create and swap MSPE patch embedding
    mspe_embed = variant_class(
        patch_size=2,
        in_chans=3,
        embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm,
        spatial_dims=2,
        K=DEFAULT_K,
        resolutions=DEFAULT_RESOLUTIONS,
    ).to(device)

    # Warm-start kernels from the pretrained baseline embedding
    if WARM_START:
        _warm_start_mspe_from_baseline(mspe_embed, baseline_checkpoint, device)

    # Only the frozen backbone comes from baseline
    model.swinViT.patch_embed = mspe_embed
    print(f"Swapped patch_embed to: {variant_class.__name__}")
    print(f"{model.swinViT.patch_embed}")

    # Freeze everything except patch_embed
    for name, param in model.named_parameters():
        if "patch_embed" not in name:
            param.requires_grad = False

    #  Verify freeze params
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    frozen_params = total_params - trainable_params
    print(f" Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
    print(f" Frozen params:    {frozen_params:,} ({100*frozen_params/total_params:.2f}%)")

    #  Verify grad flow
    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"  Trainable parameter groups ({len(trainable_names)}):")
    for n in trainable_names:
        print(f"- {n}")

    return model

In [ ]:
# Evals

# Resize for histopology
def get_aspect_preserving_target_size(inputs, effective_resolution, divisor=32):
    spatial_shape = inputs.shape[2:]
    spatial_dims = len(spatial_shape)
    current_eff = float(np.prod(spatial_shape)) ** (1.0 / spatial_dims)
    scale = effective_resolution / current_eff
    target_size = []
    for dim in spatial_shape:
        resized_dim = int(round(dim * scale))
        resized_dim = max(divisor, int(round(resized_dim / divisor)) * divisor)
        target_size.append(resized_dim)
    return target_size

# Batch resize
def resize_eval_batch(inputs, labels, roi_masks, effective_resolution):
    if effective_resolution is None:
        return inputs, labels, roi_masks
    spatial_dims = inputs.ndim - 2
    target_size = get_aspect_preserving_target_size(inputs, effective_resolution)
    image_mode = "bilinear" if spatial_dims == 2 else "trilinear"
    inputs = F.interpolate(inputs, size=target_size, mode=image_mode, align_corners=False)
    labels = F.interpolate(labels.float(), size=target_size, mode="nearest").to(labels.dtype)
    roi_masks = F.interpolate(roi_masks.float(), size=target_size, mode="nearest").to(roi_masks.dtype)
    return inputs, labels, roi_masks

# Print metrics
def print_metric_table(title, dice_vals, hd95_vals):
    """Print per class Dice + HD95."""
    num_classes = hd95_vals.shape[0]
    print(title)
    print(f"{'Class':>8} | {'Dice':>8} | {'HD95':>10}")
    print('-' * 32)
    for c in range(num_classes):
        d = dice_vals[c].item()
        h = hd95_vals[c].item()
        print(f"{c+1:>8} | {d:>8.4f} | {h:>10.2f}")
    mean_dice = dice_vals.nanmean().item()
    mean_hd95 = hd95_vals.nanmean().item()
    print('-' * 32)
    print(f"{'Mean':>8} | {mean_dice:>8.4f} | {mean_hd95:>10.2f}")
    print()
    return mean_dice, mean_hd95


def log_eval_result(prefix, result, best_metric_epoch):
    """Log per-class and mean metrics to W&B."""
    if wandb.run is None:
        return

    log_data = {
        f"{prefix}/mean_dice": result["mean_dice"],
        f"{prefix}/mean_hd95": result["mean_hd95"],
        "best_metric_epoch": best_metric_epoch,
    }
    for c, (dice_value, hd95_value) in enumerate(zip(result["dice_vals"], result["hd95_vals"]), start=1):
        log_data[f"{prefix}/class_{c}_dice"] = dice_value.item()
        log_data[f"{prefix}/class_{c}_hd95"] = hd95_value.item()
    wandb.log(log_data)

# Eval curr res
def evaluate_test_loader(model, data_loader, resolution=None, prefix="test_native", best_metric_epoch=-1):
    hd95_per_class = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean_batch")
    dice_per_class = DiceMetric(include_background=False, reduction="mean_batch")

    with torch.no_grad():
        for test_data in data_loader:
            test_inputs, test_labels, test_roi_masks = (
                test_data["image"].to(device),
                test_data["label"].to(device),
                test_data["roi_mask"].to(device),
            )
            test_inputs, test_labels, test_roi_masks = resize_eval_batch(
                test_inputs, test_labels, test_roi_masks, resolution
            )
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                test_outputs = model(test_inputs)

            test_outputs_list = [post_pred(i) for i in decollate_batch(test_outputs)]
            test_labels_list = [post_label(i) for i in decollate_batch(test_labels)]
            test_roi_masks_list = decollate_batch(test_roi_masks)

            test_outputs_list = [pred * roi.to(dtype=pred.dtype) for pred, roi in zip(test_outputs_list, test_roi_masks_list)]
            test_labels_list = [label * roi.to(dtype=label.dtype) for label, roi in zip(test_labels_list, test_roi_masks_list)]

            test_outputs_list_cpu = [v.cpu() for v in test_outputs_list]
            test_labels_list_cpu = [v.cpu() for v in test_labels_list]

            hd95_per_class(y_pred=test_outputs_list_cpu, y=test_labels_list_cpu)
            dice_per_class(y_pred=test_outputs_list_cpu, y=test_labels_list_cpu)

    hd95_vals = hd95_per_class.aggregate()
    dice_vals = dice_per_class.aggregate()
    hd95_per_class.reset()
    dice_per_class.reset()

    title = "\nNative test metrics" if resolution is None else f"Resized test metrics at effective res {resolution}"
    mean_dice, mean_hd95 = print_metric_table(title, dice_vals, hd95_vals)
    result = {
        "dice_vals": dice_vals,
        "hd95_vals": hd95_vals,
        "mean_dice": mean_dice,
        "mean_hd95": mean_hd95,
    }
    log_eval_result(prefix, result, best_metric_epoch)
    return result

# Eval all res
def evaluate_all_resolutions(model, test_loader, exp_name, best_metric_epoch, checkpoint_path=None):
    print(f"\n{'='*40}")
    print(f"TEST EVALUATION: {exp_name}")
    print(f"{'='*40}")

    ckpt_path = checkpoint_path or os.path.join(root_dir, f"best_metric_{exp_name}.pth")
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    model.eval()

    # Native res
    native_result = evaluate_test_loader(
        model, test_loader, resolution=None,
        prefix="test_native", best_metric_epoch=best_metric_epoch,
    )

    # Multi res
    multi_resolution_results = {}
    for hw in TEST_RESOLUTIONS:
        multi_resolution_results[hw] = evaluate_test_loader(
            model, test_loader, resolution=hw,
            prefix=f"test_res_{hw}", best_metric_epoch=best_metric_epoch,
        )

    
    if wandb.run is not None:
        wandb.log({
            "test/mean_dice": native_result["mean_dice"],
            "test/mean_hd95": native_result["mean_hd95"],
            "best_metric_epoch": best_metric_epoch,
        })

    return {"native": native_result, "multi_res": multi_resolution_results}

In [ ]:
# Kernels analysis

# See kernels drift
def analyze_kernel_drift(initial_state, model, exp_name):
    patch_embed = model.swinViT.patch_embed
    if not hasattr(patch_embed, "patch_kernels"):
        print(f"{exp_name}: no MSPE kernels; skipping drift analysis.")
        return None

    final_state = patch_embed.state_dict()
    metrics_l2 = []
    metrics_cos = []
    metrics_max = []

    print(f"\nKernel drift analysis: {exp_name}")
    print("Kernel | L2 Dist  | Cosine Sim | Max Diff")
    print("-" * 44)

    for k in range(len(patch_embed.patch_kernels)):
        weight_key = f"patch_kernels.{k}.weight"
        if weight_key not in initial_state or weight_key not in final_state:
            raise KeyError(f"Missing MSPE weight key for drift analysis: {weight_key}")

        w_init = initial_state[weight_key].detach().flatten().cpu()
        w_final = final_state[weight_key].detach().flatten().cpu()
        l2_dist = torch.norm(w_final - w_init, p=2).item()
        cos_sim = F.cosine_similarity(w_final.unsqueeze(0), w_init.unsqueeze(0)).item()
        max_diff = torch.max(torch.abs(w_final - w_init)).item()

        metrics_l2.append(l2_dist)
        metrics_cos.append(cos_sim)
        metrics_max.append(max_diff)
        print(f"{k:>6} | {l2_dist:>8.4f} | {cos_sim:>10.4f} | {max_diff:>8.4f}")

    drift = {
        "l2": metrics_l2,
        "cosine": metrics_cos,
        "max_diff": metrics_max,
    }

    if wandb.run is not None:
        log_data = {}
        for k, (l2_dist, cos_sim, max_diff) in enumerate(zip(metrics_l2, metrics_cos, metrics_max)):
            log_data[f"kernel_drift/kernel_{k}_l2"] = l2_dist
            log_data[f"kernel_drift/kernel_{k}_cosine"] = cos_sim
            log_data[f"kernel_drift/kernel_{k}_max_diff"] = max_diff
        wandb.log(log_data)

    x = np.arange(len(metrics_l2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(x, metrics_l2)
    axes[0].set_title("L2 distance")
    axes[0].set_xlabel("Kernel")
    axes[1].bar(x, metrics_cos)
    axes[1].set_title("Cosine similarity")
    axes[1].set_xlabel("Kernel")
    axes[2].bar(x, metrics_max)
    axes[2].set_title("Max absolute diff")
    axes[2].set_xlabel("Kernel")
    fig.suptitle(f"MSPE kernel drift: {exp_name}")
    plt.tight_layout()

    if wandb.run is not None:
        wandb.log({"kernel_drift/plot": wandb.Image(fig)})
    plt.show()

    return drift

In [ ]:
# Training

def train_variant(model, train_loader, val_loader, exp_config):
    checkpoint_path = exp_config["checkpoint_path"]
    max_epochs = exp_config["max_epochs"]

    masked_dice = MaskedDiceLoss(include_background=False, to_onehot_y=True, softmax=False)
    loss_function = lambda logits, target, mask=None: masked_dice(logits.softmax(dim=1), target, mask=mask)
    # AdamW optimizer 
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WD,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    scaler = GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

    best_metric = -1
    best_metric_epoch = -1
    epoch_loss_values = []
    metric_values = []
    epochs_no_improve = 0
    
    total_start = time.time()
    completed_epochs = 0

    for epoch in range(max_epochs):
        epoch_start = time.time()
        print("-" * 10)
        print(f"epoch {epoch + 1}/{max_epochs}")
        model.train()
        epoch_loss = 0
        step = 0

        for batch_data in train_loader:
            step += 1
            inputs, labels, roi_masks = (
                batch_data["image"].to(device),
                batch_data["label"].to(device),
                batch_data["roi_mask"].to(device),
            )

            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                # NOTE: random train resize is done inside mspe_swin_train_step
                loss = mspe_swin_train_step(model, inputs, labels, loss_function, lam=exp_config["lam"], mask=roi_masks)
            scaler.scale(loss).backward()

            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            wandb.log({"train/step_loss": loss.item(), "train/amp_scale": scaler.get_scale()})
            print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")

        epoch_loss /= step
        epoch_loss_values.append(epoch_loss)
        completed_epochs = epoch + 1
        print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
        epoch_time_sec = time.time() - epoch_start
        print(f"time per epoch: {epoch_time_sec:.2f} s")

        current_lr = optimizer.param_groups[0]['lr']
        wandb.log({
            "train/epoch_loss": epoch_loss,
            "lr": current_lr,
            "epoch": epoch + 1,
            "train/time_per_epoch": epoch_time_sec
        })

        if (epoch + 1) % VAL_INTERVAL == 0:
            model.eval()
            with torch.no_grad():
                # Validate across resolutions so the mid-resolution kernel 
                per_res_dice = {}
                for sel_res in VAL_SELECT_RESOLUTIONS:
                    dice_metric.reset()
                    for val_data in val_loader:
                        val_inputs, val_labels, val_roi_masks = (
                            val_data["image"].to(device),
                            val_data["label"].to(device),
                            val_data["roi_mask"].to(device),
                        )
                        # reuse the eval resize helper (defined in the evals cell)
                        val_inputs, val_labels, val_roi_masks = resize_eval_batch(
                            val_inputs, val_labels, val_roi_masks, sel_res
                        )
                        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                            val_outputs = model(val_inputs)

                        val_outputs_list = [post_pred(i) for i in decollate_batch(val_outputs)]
                        val_labels_list = [post_label(i) for i in decollate_batch(val_labels)]
                        val_roi_masks_list = decollate_batch(val_roi_masks)

                        val_outputs_list = [pred * roi.to(dtype=pred.dtype) for pred, roi in zip(val_outputs_list, val_roi_masks_list)]
                        val_labels_list = [label * roi.to(dtype=label.dtype) for label, roi in zip(val_labels_list, val_roi_masks_list)]
                        dice_metric(y_pred=val_outputs_list, y=val_labels_list)

                    res_key = "native" if sel_res is None else str(sel_res)
                    per_res_dice[res_key] = dice_metric.aggregate().item()
                    dice_metric.reset()

                # Select the checkpoint on the mean Dice across the chosen resolutions
                metric = float(np.mean(list(per_res_dice.values())))
                metric_values.append(metric)

                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), checkpoint_path)
                    print(f"saved new best metric model to {checkpoint_path}")
                    print(
                        f"current epoch: {epoch + 1} current mean dice (multi-res): {metric:.3f}", '\n'
                        f"per-res dice: { {k: round(v, 3) for k, v in per_res_dice.items()} }", '\n'
                        f"best validation mean dice: {best_metric:.3f} "
                        f"at epoch: {best_metric_epoch}"
                    )
                else:
                    epochs_no_improve += VAL_INTERVAL

                wandb.log({
                    "val Dice": metric,
                    **{f"val Dice res {k}": v for k, v in per_res_dice.items()},
                    "epoch": epoch + 1,
                })

                if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping triggered at epoch {epoch + 1}. No improvement for {EARLY_STOPPING_PATIENCE} epochs.")
                    wandb.log({"early_stop_epoch": epoch + 1})
                    break

        scheduler.step()
        
    total_time = time.time() - total_start
    print(f"total epochs: {completed_epochs}, total training time: {total_time / 60:.2f} min, average time per epoch: {total_time / completed_epochs:.2f} s")

    return best_metric, best_metric_epoch, epoch_loss_values, metric_values

In [ ]:
# Eval baseline checkpoint
login_wandb()
wandb.init(
    project=WANDB_PROJECT,
    name=f"BASELINE_CHECKPOINT_EVAL_{FOLD_TAG}",
    tags=[FOLD_TAG, "baseline_eval"],
    config={
        "checkpoint": BASELINE_CHECKPOINT,
        "feature_size": FEATURE_SIZE,
        "resolutions": TEST_RESOLUTIONS,
        "eval_only": True,
        "model_type": "baseline_swin_unetr",
        "use_v2": True,
        "fold_number": FOLD_NUMBER,
        "fold_tag": FOLD_TAG,
    },
)

baseline_model = create_baseline_model(BASELINE_CHECKPOINT, device)

# Run the same native + multi-res eval
baseline_results = evaluate_all_resolutions(
    baseline_model,
    test_loader,
    exp_name="BASELINE",
    best_metric_epoch=-1,
    checkpoint_path=BASELINE_CHECKPOINT,
)

# Clean up the baseline from memory
wandb.finish()
del baseline_model
torch.cuda.empty_cache()
print("Baseline checkpoint evaluation complete.")

In [ ]:
# Experiments loop

login_wandb()

# Display baseline results  with fine-tuned results
assert "baseline_results" in globals(), (
    "Run the baseline checkpoint eval cell before running the experiments loop."
)
all_results = {"BASELINE": baseline_results}
all_drift = {}
all_training_curves = {}

for exp_idx, exp in enumerate(EXPERIMENTS):
    set_determinism(seed=2026)
    
    exp_name = exp["name"]
    exp_class = exp["mspe_class"]
    exp_desc = exp["description"]

    print(f"\n{'='*60}")
    print(f"EXPERIMENT {exp_idx + 1}/{len(EXPERIMENTS)}: {exp_name}")
    print(f"  {exp_desc}")
    print(f"{'='*60}\n")

    checkpoint_path = os.path.join(root_dir, f"best_metric_{exp_name}_{FOLD_TAG}.pth")

    # wb init for this variant
    wandb.init(
        project=WANDB_PROJECT,
        name=f"{exp_name}_{FOLD_TAG}",
        save_code=True,
        group=exp_name,
        tags=[FOLD_TAG],
        config={
            "max_epochs": MAX_EPOCHS,
            "val_interval": VAL_INTERVAL,
            "batch_size": BATCH_SIZE,
            "feature_size": FEATURE_SIZE,
            "gradient_checkpointing": False,
            "learning_rate": LR,
            "weight_decay": WD,
            "loss_type": "MaskedDiceLoss",
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "mspe_variant": exp_class.__name__,
            "lambda": exp["lam"],
            "frozen_backbone": True,
            "use_v2": True,
            "baseline_checkpoint": BASELINE_CHECKPOINT,
            "fold_number": FOLD_NUMBER,
            "fold_tag": FOLD_TAG,
            "K": DEFAULT_K,
            "resolutions": DEFAULT_RESOLUTIONS,
            "checkpoint_path": checkpoint_path,
        },
    )

    # Create model
    model = create_model_for_variant(exp_class, BASELINE_CHECKPOINT, device)

    # Save initial embedding weights
    initial_mspe_state = copy.deepcopy(model.swinViT.patch_embed.state_dict())

    # Train all
    exp_config = {
        "name": exp_name,
        "max_epochs": MAX_EPOCHS,
        "checkpoint_path": checkpoint_path,
        "lam": exp["lam"],
    }
    best_metric, best_metric_epoch, epoch_losses, metric_vals = train_variant(
        model, train_loader, val_loader, exp_config
    )
    all_training_curves[exp_name] = {
        "epoch_losses": epoch_losses,
        "metric_values": metric_vals,
        "best_metric": best_metric,
        "best_metric_epoch": best_metric_epoch,
    }

    # Eval
    results = evaluate_all_resolutions(
        model, test_loader, exp_name, best_metric_epoch,
        checkpoint_path=exp_config["checkpoint_path"],
    )
    all_results[exp_name] = results

    # See embedding drift
    drift = analyze_kernel_drift(initial_mspe_state, model, exp_name)
    all_drift[exp_name] = drift

    # Clean
    wandb.finish()
    del model
    torch.cuda.empty_cache()
    print(f"\n--- Completed {exp_name}")

print("\n" + "="*60)
print("ALL EXPERIMENTS COMPLETED")
print("="*60)

In [ ]:
# Summary table

print(f"\n{'='*80}")
print(f"RESULTS SUMMARY: Frozen backbone MSPE fine tuning")
print(f"{'='*80}\n")

# Header
res_cols = ["Native"] + [str(r) for r in TEST_RESOLUTIONS]
header = f"{'Condition':<20} | " + " | ".join(f"{c:>8}" for c in res_cols) + " |"
sep = "-" * len(header)

# Dice table
print("\nDice (class 1):")
print(sep)
print(header)
print(sep)
for exp_name, results in all_results.items():
    native_dice = results["native"]["mean_dice"]
    row = f"{exp_name:<20} | {native_dice:>8.4f}"
    for hw in TEST_RESOLUTIONS:
        dice = results["multi_res"][hw]["mean_dice"]
        row += f" | {dice:>8.4f}"
    row += " |"
    print(row)
print(sep)

# HD95 table
print("\nHD95 (class 1):")
print(sep)
print(header)
print(sep)
for exp_name, results in all_results.items():
    native_hd95 = results["native"]["mean_hd95"]
    row = f"{exp_name:<20} | {native_hd95:>8.2f}"
    for hw in TEST_RESOLUTIONS:
        hd95 = results["multi_res"][hw]["mean_hd95"]
        row += f" | {hd95:>8.2f}"
    row += " |"
    print(row)
print(sep)

# Training summary
print("\nTraining Summary:")
print(f"{'Condition':<20} | {'Best Dice':>10} | {'Best Epoch':>10} | {'Final Loss':>10}")
print("-" * 60)
for exp_name, curves in all_training_curves.items():
    print(f"{exp_name:<20} | {curves['best_metric']:>10.4f} | {curves['best_metric_epoch']:>10d} | {curves['epoch_losses'][-1]:>10.4f}")

In [ ]:
# Train curves

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loss curves
ax = axes[0]
for exp_name, curves in all_training_curves.items():
    epochs = list(range(1, len(curves["epoch_losses"]) + 1))
    ax.plot(epochs, curves["epoch_losses"], label=exp_name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss (all variants)")
ax.legend()
ax.grid(alpha=0.3)

# Val Dice curves
ax = axes[1]
for exp_name, curves in all_training_curves.items():
    val_epochs = [VAL_INTERVAL * (i + 1) for i in range(len(curves["metric_values"]))]
    ax.plot(val_epochs, curves["metric_values"], label=exp_name, marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Dice")
ax.set_title("Validation Dice (all variants)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quick sanity check for all variats

def to_display_image(tensor_chw):
    image = tensor_chw.detach().cpu()
    if image.shape[0] in (3, 4):
        return image.permute(1, 2, 0)
    return image[0]

vis_resolutions = [None] + list(TEST_RESOLUTIONS)

for exp in EXPERIMENTS:
    ckpt = os.path.join(root_dir, f"best_metric_{exp['name']}_{FOLD_TAG}.pth")
    print(f"Visualizing {exp['name']} from {ckpt}")

    model_viz = create_model_for_variant(exp["mspe_class"], BASELINE_CHECKPOINT, device)
    model_viz.load_state_dict(torch.load(ckpt, weights_only=True))
    model_viz.eval()

    with torch.no_grad():
        for i, val_data in enumerate(val_loader):
            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)
            val_roi_masks = val_data["roi_mask"].to(device)
            n_cols = len(vis_resolutions)
            fig, axes_vis = plt.subplots(4, n_cols, figsize=(5 * n_cols, 16))
            if n_cols == 1:
                axes_vis = axes_vis.reshape(4, 1)
            for col, effective_resolution in enumerate(vis_resolutions):
                inputs_r, labels_r, roi_r = resize_eval_batch(
                    val_inputs, val_labels, val_roi_masks, effective_resolution)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    outputs = model_viz(inputs_r)
                image = to_display_image(inputs_r[0])
                label = labels_r[0, 0].detach().cpu()
                roi_mask = roi_r[0, 0].detach().cpu()
                pred = torch.argmax(outputs, dim=1).detach().cpu()[0]
                pred = pred * roi_mask.to(dtype=pred.dtype)
                res_title = "Native" if effective_resolution is None else f"Res {effective_resolution}"
                axes_vis[0, col].imshow(image, cmap="gray")
                axes_vis[0, col].set_title(f"Image — {res_title}")
                axes_vis[0, col].axis("off")
                axes_vis[1, col].imshow(roi_mask, cmap="gray")
                axes_vis[1, col].set_title(f"ROI — {res_title}")
                axes_vis[1, col].axis("off")
                axes_vis[2, col].imshow(label * roi_mask, cmap="viridis")
                axes_vis[2, col].set_title(f"Label — {res_title}")
                axes_vis[2, col].axis("off")
                axes_vis[3, col].imshow(pred, cmap="viridis")
                axes_vis[3, col].set_title(f"Prediction — {res_title}")
                axes_vis[3, col].axis("off")
            plt.suptitle(f"Validation sample {i} — {exp['name']}")
            plt.tight_layout()
            plt.show()
            if i == 2:
                break

    del model_viz
    torch.cuda.empty_cache()

In [ ]:
# !pip freeze > /content/drive/MyDrive/MSPE/requirements_ft_frozen.txt

In [ ]:
# Disconnect from runtime 
runtime.unassign()